# 00 Predictive vs Causal: DAGs and Omitted Variable Bias

Up to now you have answered one question: 'how well does $x$ predict $y$?' This notebook is about a different question: 'if we *change* $x$, what happens to $y$?' Those are different questions and OLS coefficients answer the first one without intervention.


## Table of Contents
- [Two questions, one regression table](#two-questions)
- [DAGs in 5 minutes](#dags)
- [Omitted variable bias, simulated](#ovb)
- [The collider trap (bad controls)](#collider)
- [Good controls, bad controls, neutral controls](#control-zoo)
- [Apply to a macro question](#macro-application)
- [Checkpoint (Self-Check)](#checkpoint-self-check)
- [Solutions (Reference)](#solutions-reference)


## Why This Notebook Matters
Every regression you have run so far has been *predictive*. The coefficient on `T10Y2Y_lag1` tells you the slope that minimizes prediction error in your data. It does **not** tell you what would happen if the Fed engineered a steeper yield curve next quarter.

Most macro and applied-econ work that matters is causal: 'if we raise the minimum wage, what happens to employment?' 'If the Fed cuts rates 50bps, what happens to GDP?' Answering those questions requires a different toolkit, and the first piece is being able to *see* when a coefficient is and isn't causal.

You will learn:
- the difference between observational and interventional questions,
- DAG (directed acyclic graph) notation as a back-of-envelope tool for thinking about causation,
- omitted variable bias by simulation: when does dropping a variable bias the coefficient on the variable you kept?
- the collider problem: why adding a variable can *introduce* bias instead of removing it,
- a working taxonomy of good, bad, and neutral controls.

## Prerequisites (Quick Self-Check)
- §02 (regression) — comfortable with multiple-regression coefficient interpretation.
- §02b/§03 ML/classification helpful but not required.
- A willingness to entertain the idea that a 'better R²' can mean a *worse* causal answer.

## What You Will Produce
- A simulation that recovers a known causal effect by including the right controls.
- A simulation that biases the same effect by including a *bad* control.
- A short DAG you draw (in markdown) for one macro-econ question of your choosing.

## Common Pitfalls
- 'I controlled for everything I could think of' — adding more controls is not always better.
- Treating large $R^2$ as evidence of a causal model. They are unrelated.
- Claiming a causal interpretation in a write-up while only having shown predictive evidence.
- Using lagged predictors as if that automatically established causation. Lagging helps with simultaneity, not with confounding.

## Quick Fixes (When You Get Stuck)
- If your simulated coefficient does not look biased, try larger sample sizes or stronger confounder effects.
- If a 'good control' makes things worse, double-check whether it is a *descendant* of the treatment (collider) rather than a confounder.

## Matching Guide
- `docs/guides/05_causal_inference/00_predictive_vs_causal_dags.md`


<a id="environment-bootstrap"></a>
## Environment Bootstrap


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    p = start
    for _ in range(8):
        if (p / 'src').exists() and (p / 'docs').exists():
            return p
        p = p.parent
    raise RuntimeError('Could not find repo root. Start Jupyter from the repo root.')


PROJECT_ROOT = find_repo_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT


<a id="two-questions"></a>
## Two questions, one regression table

Suppose you regress GDP growth on the yield-curve spread and get $\hat\beta = 0.4$ with a t-stat of 5. The number 0.4 answers the question:

> **Predictive:** Among the historical observations in this dataset, when the spread is one percentage point wider, GDP growth tends to be 0.4 percentage points higher in the next quarter, after fitting the constant.

It does **not** answer:

> **Causal:** If we somehow forced the spread to widen by one percentage point, GDP growth would rise by 0.4 percentage points in the next quarter.

These questions can have wildly different answers because the spread is not assigned randomly. Things that move the spread (Fed policy, inflation expectations, recession fears) also move GDP growth through other channels. The 0.4 in your regression bundles the direct effect with all of those side channels.

Answering causal questions requires *identification*: a story for why your coefficient really does isolate the intervention you care about. The DAG framework is one disciplined way to write that story down before doing any math.


<a id="dags"></a>
## DAGs in 5 minutes

A **DAG** (directed acyclic graph) is a picture of your causal assumptions. Variables are nodes; arrows are direct causal effects.

Three building blocks:

**Chain:** $A \to B \to C$. $B$ is a *mediator*. If you regress $C$ on $A$, you measure the total effect of $A$ on $C$. If you also include $B$, you measure only the *direct* effect not going through $B$.

**Fork (confounder):** $A \leftarrow C \to B$. $C$ is a *common cause*. Regressing $B$ on $A$ alone confounds the relationship; controlling for $C$ removes the confounding.

**Collider:** $A \to C \leftarrow B$. $C$ is a *common effect*. Controlling for $C$ *induces* a spurious correlation between $A$ and $B$ even when they are independent. This is the 'bad control' trap.

Two rules of thumb you can use immediately:

1. **Control for confounders.** Include any variable that is a common cause of your treatment and outcome.
2. **Do not control for colliders or post-treatment variables.** They open backdoor paths or block direct effects.

The hard part is figuring out which of your candidate controls is which. You cannot tell from the data alone — that information has to come from theory or domain knowledge.


### Your Turn (1): Draw a DAG for a macro question

Pick one of these and write the DAG in markdown using arrows like `A -> B`:
1. Effect of the Fed Funds rate on inflation, with the unemployment rate possibly involved.
2. Effect of yield-curve spread on GDP growth, with monetary policy as a potential confounder.
3. Your own pick — write down what you think the variables are and how they connect.

There is no 'right' answer. The point is to commit to a structure and notice what controlling for which variable would do.


In [ ]:
my_dag = """
Question: ...

Variables:
- T (treatment) = ...
- Y (outcome) = ...
- C1, C2 = ...

Edges:
- ... -> ...
- ... -> ...

Controls I would include and why: ...
Controls I would exclude and why: ...
"""
print(my_dag)


<a id="ovb"></a>
## Omitted variable bias, simulated

Omitted variable bias is a fork: you have $T \leftarrow C \to Y$, $T$ is your treatment, $Y$ is your outcome, $C$ is the common cause, and you mistakenly leave $C$ out of the regression.

The classic formula: in a regression of $Y$ on $T$ alone, the coefficient picks up the true causal effect *plus* a bias term equal to (effect of $C$ on $Y$) × (regression of $T$ on $C$).

We will simulate this so you can see exactly how it works.


### Your Turn (1): Generate confounded data


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

rng = np.random.default_rng(0)
n = 5000

# Confounder C affects both T and Y
C = rng.normal(size=n)

# Treatment T = 0.8*C + noise (so T is correlated with C)
T = 0.8 * C + rng.normal(scale=0.5, size=n)

# Outcome Y = 1.0*T + 1.5*C + noise. The true causal effect of T on Y is 1.0.
Y = 1.0 * T + 1.5 * C + rng.normal(scale=0.5, size=n)

df = pd.DataFrame({'Y': Y, 'T': T, 'C': C})
df.head()


### Your Turn (2): Naive regression vs adjusted regression


In [ ]:
# Naive: omit the confounder
X1 = sm.add_constant(df[['T']], has_constant='add')
naive = sm.OLS(df['Y'], X1).fit()

# Adjusted: include the confounder
X2 = sm.add_constant(df[['T', 'C']], has_constant='add')
adj = sm.OLS(df['Y'], X2).fit()

comparison = pd.DataFrame({
    'naive (Y ~ T)': naive.params,
    'adjusted (Y ~ T + C)': adj.params,
}).round(4)
comparison


### Your Turn (3): Sanity-check the bias formula

The bias in a single-regressor regression is:

$$
\text{bias} = \beta_C \cdot \frac{\mathrm{Cov}(T, C)}{\mathrm{Var}(T)}
$$

That second term is the slope you get when you regress **C on T** (not T on C — direction matters here, because we want the share of $C$'s variation that the included regressor $T$ already captures). Compute both pieces and check.


In [ ]:
# TODO: regress C on T (not T on C) to get the slope; compute the implied bias.
X_CT = sm.add_constant(df[['T']], has_constant='add')
C_on_T = sm.OLS(df['C'], X_CT).fit()
slope_C_on_T = ...

# 'Effect of C on Y holding T fixed' is the C coefficient from the adjusted regression.
effect_C_on_Y_adj = ...

implied_bias = slope_C_on_T * effect_C_on_Y_adj
naive_T_coef = naive.params['T']
true_T_effect = 1.0
print(f'Naive T coef:       {naive_T_coef:.4f}')
print(f'True effect:        {true_T_effect:.4f}')
print(f'Empirical bias:     {naive_T_coef - true_T_effect:.4f}')
print(f'Implied by formula: {implied_bias:.4f}')


<a id="collider"></a>
## The collider trap (bad controls)

OVB tells you to include the right controls. The collider problem tells you that some 'controls' are *not* the right ones — including them *creates* bias instead of removing it.

DAG: $T \to M \leftarrow Y$. $M$ is a common effect of treatment and outcome. Controlling for $M$ opens a non-causal path between $T$ and $Y$ and biases the coefficient.


### Your Turn (1): Simulate a collider


In [ ]:
rng = np.random.default_rng(1)
n = 5000

# T and Y are independent (no causal effect of T on Y at all)
T = rng.normal(size=n)
Y = rng.normal(size=n)

# M is a common effect of T and Y
M = 1.0 * T + 1.0 * Y + rng.normal(scale=0.5, size=n)

df_c = pd.DataFrame({'Y': Y, 'T': T, 'M': M})

# Regress Y on T alone (correct: should find no effect)
naive_c = sm.OLS(df_c['Y'], sm.add_constant(df_c[['T']], has_constant='add')).fit()
# Regress Y on T and M (incorrect: M is a collider)
bad_c = sm.OLS(df_c['Y'], sm.add_constant(df_c[['T', 'M']], has_constant='add')).fit()

pd.DataFrame({
    'correct (Y ~ T)': naive_c.params,
    'collider-controlled (Y ~ T + M)': bad_c.params,
}).round(4)


### Your Turn (2): Interpret

The 'correct' regression should give a T coefficient near 0 (true effect is 0). The 'collider-controlled' regression gives a non-zero T coefficient because conditioning on $M$ created a relationship between $T$ and $Y$ that does not exist in the data-generating process.

Write 4–6 sentences:
- Why does adding more controls *not* automatically make causal inference more credible?
- What domain knowledge would have warned you that $M$ is a collider rather than a confounder?


In [ ]:
notes = """
...
"""
print(notes)


<a id="control-zoo"></a>
## Good controls, bad controls, neutral controls

Cinelli, Forney, and Pearl (2022) gave a clean taxonomy. Quick summary, where $T$ is treatment and $Y$ is outcome:

| Variable type | Position in DAG | Effect on causal estimate of $T$ on $Y$ |
|---|---|---|
| **Confounder** | $T \leftarrow X \to Y$ | Including $X$ removes bias. **Include.** |
| **Mediator** | $T \to X \to Y$ | Including $X$ blocks the indirect causal path; you measure only the direct effect. **Include only if that's what you want.** |
| **Collider** | $T \to X \leftarrow Y$ | Including $X$ creates spurious association. **Exclude.** |
| **Descendant of $Y$** | $Y \to X$ | Equivalent to controlling for $Y$; absurd. **Exclude.** |
| **Descendant of $T$ (post-treatment)** | $T \to X$ | Often a partial mediator; usually biases. **Exclude unless intentional.** |
| **Pure predictor of $Y$** | $X \to Y$ only | Reduces residual variance, narrows CI; does not affect identification. **Optional, helpful.** |
| **Pure predictor of $T$** | $X \to T$ only | Does not bias $T$'s coefficient. **Neutral.** |

Everything you 'control for' must come from category 1 or category 6 to get a clean causal estimate.


<a id="macro-application"></a>
## Apply to a macro question

Take one specific question:

> Does the yield-curve spread (`T10Y2Y`) cause future GDP growth?

Sketch a DAG. Likely nodes:
- `T10Y2Y` — treatment.
- `gdp_growth_qoq_next` — outcome.
- `FEDFUNDS` — Fed policy. Common cause? Mediator? Both?
- `CPI inflation expectations` (latent; not in dataset).
- `recession sentiment / forward-looking sentiment` (latent).

Write the DAG in your own words. Then write down which variables in our dataset you would include as controls if you wanted a causal answer, and which you would deliberately *not* include because they look like mediators or post-treatment variables.


In [ ]:
macro_dag = """
DAG sketch:
...

Controls I would include (and why each is a confounder, not a mediator/collider):
...

Controls I would exclude (and why each is a mediator, collider, or descendant):
...

Bottom line on whether the yield-curve regression coefficient is causal:
...
"""
print(macro_dag)


<a id="checkpoint-self-check"></a>
## Checkpoint (Self-Check)


In [ ]:
# Sanity asserts (uncomment after the cells above run cleanly):
# assert abs(adj.params['T'] - 1.0) < 0.05  # adjusted regression recovers the true effect
# assert abs(naive.params['T'] - 1.0) > 0.5  # naive regression is biased
# assert abs(bad_c.params['T']) > 0.1  # collider control creates spurious effect
...


## Extensions (Optional)
- Vary the strength of the confounder $C$ (try effects 0.1, 1.0, 5.0). Plot the OVB as a function of confounder strength.
- Modify the collider simulation so $T$ has a small true effect on $Y$ ($\beta = 0.3$). Show that the collider control still biases the estimate.
- Pick a real applied-econ paper from a journal and identify which variables they treat as confounders. Do you agree?
- Implement a simple sensitivity analysis: how strong would an unmeasured confounder need to be to overturn your finding? (See: 'E-value' calculations.)


## Reflection
- Take one regression you have run earlier in this curriculum. Was it a predictive question or a causal one? Were you treating it like the wrong kind?
- The next notebook covers difference-in-differences — a research design that gives you causal leverage when randomization is impossible. What questions in your work could it apply to?


<a id="solutions-reference"></a>
## Solutions (Reference)

<details><summary>Solution: Bias formula sanity check</summary>

```python
slope_C_on_T = C_on_T.params['T']
effect_C_on_Y_adj = adj.params['C']
implied_bias = slope_C_on_T * effect_C_on_Y_adj
```

On the simulated data with `seed=0`, `n=5000` you should see the implied and empirical bias agree closely (both ~1.33). The naive coefficient is ~2.33; the adjusted coefficient is ~1.01 (very close to the true effect of 1.0).

</details>

<details><summary>Solution: Macro DAG sketch</summary>

A reasonable sketch (yours may differ):

```
FEDFUNDS -> T10Y2Y
FEDFUNDS -> GDP_growth
T10Y2Y -> GDP_growth
(latent) recession_sentiment -> T10Y2Y
(latent) recession_sentiment -> GDP_growth
```

Implications: `FEDFUNDS` is a confounder (controlled in §02); recession sentiment is an unmeasured confounder (omitted). The OLS coefficient on `T10Y2Y_lag1` is therefore not a clean causal estimate even when controlling for `FEDFUNDS_lag1` — it is closer to causal than the bivariate version, but still biased by the unmeasured channel.

Bottom line: predictive interpretation is defensible; causal interpretation requires either a quasi-experiment (event study around a Fed announcement) or an instrumental variable.

</details>
